# Задание 2. Адаптивная реконструкция поверхностей по облакам точек ТЛО

**Вариант:** папка `5` — 500 облаков точек (`valve_XXXX_lidar_classes.ply`).

**Формат точки:** `x y z scalar_Label`, где `scalar_Label` — номер сегмента (задан заранее).

**Режимы:** `FAST_MODE = True` — быстрый демо-прогон (~15–25 мин). `FAST_MODE = False` — все 500 файлов (медленнее, но стабильно за счёт лимитов RAM). После смены режима **перезапусти kernel** и выполни все ячейки заново.

**Pipeline (по ТЗ):**
1. Загрузка данных и проверка корректности
2. Предобработка (фильтрация шума, нормализация)
3. Формирование сегментов **по `label`**
4. Геометрический анализ сегмента
5. Классификация: `plane` / `tube` / `sphere` / `complex`
6. Выбор метода реконструкции (Poisson / Alpha Shape / Ball Pivoting)
7. Реконструкция каждого сегмента
8. Сборка итоговой модели
9. Оценка качества (RMSE, Hausdorff, Chamfer, артефакты, связность)

**Предупреждение про TCP/шифрование в логе Jupyter — нормальное, не причина падения.**


## 1. Импорты и пути

In [ ]:
%pip install open3d numpy matplotlib scikit-learn scipy plyfile tqdm

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import json
import time
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import KDTree, cKDTree
from plyfile import PlyData
from tqdm import tqdm

warnings.filterwarnings("ignore")

# --- назначенная папка варианта (лаба 5) ---
CLOUDS_DIR = Path("5")
if not CLOUDS_DIR.exists():
    CLOUDS_DIR = Path("sem2/5")          # фолбэк, если запуск из корня репозитория
CLOUDS_DIR = CLOUDS_DIR.resolve()

RESULT_DIR = Path("outputs_task2")
RESULT_DIR.mkdir(exist_ok=True)
MESHES_DIR = RESULT_DIR / "meshes"
MESHES_DIR.mkdir(exist_ok=True)

# --- режим запуска ---
# FAST_MODE=True  -> ~15–25 мин на CPU (для отчёта/демо)
# FAST_MODE=False -> все 500 облаков (несколько часов)
FAST_MODE = True

if FAST_MODE:
    PROCESS_ALL = False
    DEBUG_MAX_FILES = 5            # меньше файлов — стабильнее в Jupyter
    SAVE_MESH_LIMIT = 2
    MAX_POINTS_PER_CLOUD = 3000
    MAX_POINTS_PER_SEGMENT = 1200  # Poisson/Ball Pivoting жрут RAM
    MIN_SEGMENT_POINTS = 100
    POISSON_DEPTH = 5
    POISSON_DEPTH_COMPLEX = 6
    SKIP_METHOD_COMPARISON = True
    SKIP_VISUALIZATION = True
    SKIP_VISUALIZATION = True
    ORIENT_NORMALS = False         # частая причина kernel crash в Open3D
else:
    # Полный прогон: все 500 файлов, но с лимитами RAM (иначе Jupyter падает)
    PROCESS_ALL = True
    DEBUG_MAX_FILES = 5
    SAVE_MESH_LIMIT = 10
    MAX_POINTS_PER_CLOUD = 10000
    MAX_POINTS_PER_SEGMENT = 1500
    MIN_SEGMENT_POINTS = 50
    POISSON_DEPTH = 6
    POISSON_DEPTH_COMPLEX = 7
    SKIP_METHOD_COMPARISON = True
    SKIP_VISUALIZATION = True
    ORIENT_NORMALS = False

print("Датасет:", CLOUDS_DIR)
print("FAST_MODE:", FAST_MODE, "| файлов:", "все" if PROCESS_ALL else DEBUG_MAX_FILES)

## 2. Чтение и предобработка облака

In [ ]:
def _read_ply_ascii(path):
    '''Резервный парсер ASCII PLY: x y z + метка сегмента.'''
    with open(path, "r", errors="ignore") as fh:
        props, header_ok = [], False
        for line in fh:
            s = line.strip()
            if s.startswith("property"):
                props.append(s.split()[-1])
            if s == "end_header":
                header_ok = True
                break
        if not header_ok:
            raise ValueError("нет end_header")
        raw = np.loadtxt(fh)
    if raw.ndim == 1:
        raw = raw.reshape(1, -1) if raw.size >= 4 else raw.reshape(0, 4)
    if raw.shape[0] == 0 or raw.shape[1] < 4:
        raise ValueError("нет точек")
    low = [p.lower() for p in props]
    ix, iy, iz = low.index("x"), low.index("y"), low.index("z")
    label_key = next((p for p in ("scalar_Label", "label", "scalar_label")
                      if p.lower() in low), None)
    if label_key is None:
        raise ValueError(f"нет label в свойствах: {props}")
    il = low.index(label_key.lower())
    coords = raw[:, [ix, iy, iz]].astype(np.float64)
    labels = raw[:, il].astype(np.int64)
    return coords, labels


def read_ply_cloud(path):
    '''Читает PLY: x, y, z и метка сегмента (label / scalar_Label).

    Обрабатывает ошибки; для битых файлов бросает исключение.'''
    path = Path(path)
    try:
        data = PlyData.read(str(path))
        el = data.elements[0].data
        names = el.dtype.names or ()
        if not all(c in names for c in ("x", "y", "z")):
            raise ValueError(f"нет x/y/z: {names}")
        coords = np.stack([el["x"], el["y"], el["z"]], axis=1).astype(np.float64)
        label_key = next((k for k in ("scalar_Label", "label", "scalar_label") if k in names), None)
        if label_key is None:
            low = {n.lower(): n for n in names}
            label_key = next((low[c] for c in ("scalar_label", "label") if c in low), None)
        if label_key is None:
            raise ValueError(f"нет label: {names}")
        labels = np.asarray(el[label_key]).astype(np.int64)
    except Exception:
        coords, labels = _read_ply_ascii(path)
    if len(coords) != len(labels):
        raise ValueError("число точек и меток не совпадает")
    if len(coords) == 0:
        raise ValueError("пустое облако")
    return coords, labels


def maybe_subsample(coords, labels, max_points=None, seed=42):
    '''Опциональный downsampling с сохранением пар (точка, label).'''
    if max_points is None or len(coords) <= max_points:
        return coords, labels
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(coords), max_points, replace=False)
    return coords[idx], labels[idx]


def clean_and_normalize(coords, labels, nb_neighbors=20, std_ratio=2.0):
    '''Убирает статистические выбросы, центрирует и вписывает в единичную сферу.

    Метки переиндексируются согласованно с оставшимися точками.'''
    coords, labels = maybe_subsample(coords, labels, max_points=MAX_POINTS_PER_CLOUD)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(coords)
    pcd, keep = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)

    coords_kept = np.asarray(pcd.points)
    labels_kept = labels[keep]
    if len(coords_kept) == 0:
        raise ValueError("после фильтрации не осталось точек")

    center = coords_kept.mean(axis=0)
    coords_kept -= center
    radius = np.linalg.norm(coords_kept, axis=1).max()
    if radius > 0:
        coords_kept /= radius
    return coords_kept, labels_kept, center, radius



def subsample_segment(seg, max_points=None, seed=0):
    # Ограничивает число точек в сегменте перед тяжёлой реконструкцией.
    max_points = MAX_POINTS_PER_SEGMENT if max_points is None else max_points
    if max_points is None or len(seg) <= max_points:
        return seg
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(seg), max_points, replace=False)
    return seg[idx]


def discover_cloud_files(data_dir, validate_all=False):
    """Список .ply. По умолчанию не читает все 500 файлов (экономит RAM/OMP)."""
    all_files = sorted(Path(data_dir).glob("*.ply"))
    if not all_files:
        print("Всего .ply: 0 | валидных: 0 | пропущено: 0")
        return []

    if not validate_all:
        check = [all_files[0]]
        if len(all_files) > 1:
            check.append(all_files[-1])
        for p in check:
            coords, labels = read_ply_cloud(p)
            if len(coords) < 10:
                raise ValueError(f"{p.name}: слишком мало точек")
        print(f"Всего .ply: {len(all_files)} | валидных: {len(all_files)} "
              f"(быстрая проверка {len(check)} файлов)")
        return all_files

    valid, skipped = [], []
    for p in all_files:
        try:
            coords, labels = read_ply_cloud(p)
            if len(coords) < 10:
                raise ValueError("слишком мало точек")
            valid.append(p)
        except Exception as err:
            skipped.append((p.name, str(err)))
    print(f"Всего .ply: {len(all_files)} | валидных: {len(valid)} | пропущено: {len(skipped)}")
    if skipped:
        print("Примеры пропусков:", skipped[:3])
    return valid


cloud_files = discover_cloud_files(CLOUDS_DIR)
if cloud_files:
    xyz0, lab0 = read_ply_cloud(cloud_files[0])
    print("Пример:", cloud_files[0].name,
          "| точек:", len(xyz0),
          "| сегменты:", sorted(set(lab0.tolist())))

## 3. Разбиение на сегменты по `label`

In [ ]:
def segments_by_label(coords, labels, min_points=50):
    '''Группирует точки по значению label; слишком мелкие группы отбрасывает.'''
    result = {}
    for lbl in np.unique(labels):
        group = coords[labels == lbl]
        if len(group) >= min_points:
            result[int(lbl)] = group
    return result

## 4. Геометрический анализ сегмента

Признаки строятся на собственных значениях ковариационной матрицы (PCA) и kNN-окрестностях.
Linearity / Planarity / Sphericity (Demantke et al., 2011) — классический набор дескрипторов для облаков точек.

In [ ]:
def extract_geometry_features(seg):
    '''Геометрические дескрипторы сегмента.

    Возвращает linearity, planarity, sphericity, плотность, кривизну,
    согласованность нормалей и число связных компонент.'''
    feats = {}

    # Для больших сегментов считаем дескрипторы на подвыборке
    if len(seg) > 800:
        rng = np.random.default_rng(0)
        seg = seg[rng.choice(len(seg), 800, replace=False)]

    # PCA по всем точкам сегмента
    cov = np.cov(seg.T)
    lam = np.sort(np.linalg.eigvalsh(cov))[::-1]          # lam1 >= lam2 >= lam3
    lam = np.maximum(lam, 1e-12)
    linearity = (lam[0] - lam[1]) / lam[0]
    planarity = (lam[1] - lam[2]) / lam[0]
    sphericity = lam[2] / lam[0]
    feats.update(linearity=linearity, planarity=planarity,
                 sphericity=sphericity, eigvals=lam)

    # Плотность: среднее расстояние до ближайших соседей
    k = min(6, len(seg))
    if k >= 2:
        nbrs = NearestNeighbors(n_neighbors=k).fit(seg)
        dists, _ = nbrs.kneighbors(seg)
        feats["density"] = float(dists[:, 1:].mean())
        feats["density_std"] = float(dists[:, 1:].std())
    else:
        feats["density"] = 0.05
        feats["density_std"] = 0.0

    # Локальная кривизна: lam3 / sum(lam) в окрестности каждой точки
    if len(seg) >= 10:
        kk = min(15, len(seg) - 1)
        nbrs = NearestNeighbors(n_neighbors=kk + 1).fit(seg)
        _, idx = nbrs.kneighbors(seg)
        curvature = []
        for i in range(len(seg)):
            local = seg[idx[i, 1:]]
            ev = np.sort(np.linalg.eigvalsh(np.cov(local.T)))[::-1]
            ev = np.maximum(ev, 1e-12)
            curvature.append(ev[2] / ev.sum())
        curvature = np.array(curvature)
        feats["curvature_mean"] = float(curvature.mean())
        feats["curvature_std"] = float(curvature.std())
    else:
        feats["curvature_mean"] = 0.0
        feats["curvature_std"] = 0.0

    # Согласованность нормалей: средний |cos| угла между нормалью и нормалями соседей
    if len(seg) >= 10:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(seg)
        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamKNN(knn=min(15, len(seg) - 1)))
        normals = np.asarray(pcd.normals)
        kk = min(8, len(seg) - 1)
        nbrs = NearestNeighbors(n_neighbors=kk + 1).fit(seg)
        _, idx = nbrs.kneighbors(seg)
        cos_total, pairs = 0.0, 0
        for i in range(len(seg)):
            for j in idx[i, 1:]:
                cos_total += abs(float(normals[i] @ normals[j]))
                pairs += 1
        feats["normal_consistency"] = cos_total / max(pairs, 1)
    else:
        feats["normal_consistency"] = 0.0

    # Связность: число компонент графа соседей (union-find)
    if len(seg) >= 10:
        tree = cKDTree(seg)
        edges = tree.query_pairs(r=feats["density"] * 2.5)
        parent = list(range(len(seg)))

        def root(a):
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a

        for a, b in edges:
            ra, rb = root(a), root(b)
            if ra != rb:
                parent[ra] = rb
        feats["n_components"] = len({root(i) for i in range(len(seg))})
    else:
        feats["n_components"] = 1

    return feats

## 5. Определение типа сегмента

Решающее правило по `linearity` / `planarity` / `sphericity` плюс `normal_consistency`.
Согласованность нормалей добавлена к чисто-PCA-порогу, потому что она надёжнее отделяет
гладкие поверхности от зашумлённых фрагментов.

In [ ]:
def infer_segment_type(feats):
    lin = feats["linearity"]
    pla = feats["planarity"]
    sph = feats["sphericity"]
    nc = feats["normal_consistency"]
    curv = feats["curvature_mean"]

    # плоскость: высокая планарность, согласованные нормали, малая кривизна
    if pla > 0.5 and nc > 0.85 and curv < 0.05:
        return "plane"
    # труба / цилиндр: вытянутость по одной оси, нормали согласованы умеренно
    if lin > 0.5 and 0.5 < nc < 0.95:
        return "tube"
    # сфера: изотропность (sphericity не мала, planarity не доминирует),
    # нормали наружу -> умеренная корреляция между соседями
    if sph > 0.25 and pla < 0.5 and lin < 0.5 and 0.4 < nc < 0.85:
        return "sphere"
    return "complex" 

## 6. Подбор алгоритма реконструкции

In [ ]:
def pick_reconstruction(seg_type, feats):
    '''Возвращает (имя_метода, параметры). Параметры зависят от плотности сегмента.'''
    d = feats["density"]
    if seg_type == "plane":
        # плоскости — Alpha Shape (короткие рёбра, нет "раздувания")
        return "alpha_shape", {"alpha": max(d * 2.5, 0.01)}
    if seg_type == "tube":
        # трубчатые — Ball Pivoting с набором радиусов
        return "ball_pivoting", {"radii": [d * 1.5, d * 2.5, d * 4.0]}
    if seg_type == "sphere":
        # сферические — Poisson (хорошо замыкает поверхность)
        return "poisson", {"depth": POISSON_DEPTH}
    # сложные — Poisson с большей глубиной
    return "poisson", {"depth": POISSON_DEPTH_COMPLEX}

## 7. Реконструкция сегмента

In [ ]:
def build_mesh(seg, method, params):
    # Применяет выбранный алгоритм. Возвращает open3d.TriangleMesh либо None.
    seg = subsample_segment(np.asarray(seg, dtype=np.float64))
    if len(seg) < MIN_SEGMENT_POINTS:
        return None

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(seg)
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=20))
    if ORIENT_NORMALS:
        try:
            pcd.orient_normals_consistent_tangent_plane(k=10)
        except Exception:
            pass

    try:
        if method == "poisson":
            mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
                pcd, depth=params["depth"], linear_fit=False)
            dens = np.asarray(densities)
            if len(dens) > 0:
                mesh.remove_vertices_by_mask(dens < np.quantile(dens, 0.05))
            return mesh
        if method == "alpha_shape":
            return o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(
                pcd, params["alpha"])
        if method == "ball_pivoting":
            return o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
                pcd, o3d.utility.DoubleVector(params["radii"]))
    except Exception as err:
        warnings.warn(f"Не удалась реконструкция ({method}): {err}")
    return None


## 8. Метрики качества mesh

Считаем:
- **RMSE** — среднеквадратичная ошибка расстояния от исходных точек до поверхности.
- **Hausdorff (95-й перцентиль)** — устойчивая оценка максимального отклонения.
- **Chamfer (двусторонний)** — учитывает направления точки→mesh и mesh→точки.
- **n_artifacts** — количество несвязных компонент mesh (>1 означает фрагментацию).
- **watertight** — замкнута ли поверхность.

In [ ]:
EMPTY_METRICS = {
    "rmse": float("inf"), "hausdorff_p95": float("inf"), "chamfer": float("inf"),
    "n_artifacts": -1, "watertight": False, "n_vertices": 0, "n_triangles": 0,
}


def score_mesh(source_points, mesh):
    '''Метрики качества для пары (исходные точки, mesh).'''
    out = dict(EMPTY_METRICS)
    if mesh is None or len(mesh.vertices) == 0:
        return out

    pts = source_points
    if len(pts) > 2000:
        rng = np.random.default_rng(0)
        pts = pts[rng.choice(len(pts), 2000, replace=False)]
    n_target = min(max(int(len(pts)), 200), 2000)
    sampled = mesh.sample_points_uniformly(number_of_points=n_target)
    sampled_pts = np.asarray(sampled.points)
    if len(sampled_pts) == 0:
        return out

    # расстояния точки -> mesh и mesh -> точки
    d_p2m, _ = KDTree(sampled_pts).query(pts)
    d_m2p, _ = KDTree(pts).query(sampled_pts)

    out["rmse"] = float(np.sqrt(np.mean(d_p2m ** 2)))
    out["hausdorff_p95"] = float(max(np.quantile(d_p2m, 0.95), np.quantile(d_m2p, 0.95)))
    out["chamfer"] = float(np.mean(d_p2m) + np.mean(d_m2p))

    # cluster_connected_triangles / is_watertight иногда валят Open3D при серии файлов
    out["n_artifacts"] = 1
    out["watertight"] = False

    out["n_vertices"] = int(np.asarray(mesh.vertices).shape[0])
    out["n_triangles"] = int(np.asarray(mesh.triangles).shape[0])
    return out

## 9. Объединение в итоговую модель

In [ ]:
def combine_meshes(mesh_list):
    merged = o3d.geometry.TriangleMesh()
    for m in mesh_list:
        if m is not None and len(m.vertices) > 0:
            merged += m
    merged = merged.merge_close_vertices(1e-6)
    merged = merged.remove_degenerate_triangles()
    merged = merged.remove_unreferenced_vertices()
    merged = merged.remove_duplicated_triangles()
    return merged

## 10. Полный pipeline для одного файла

In [ ]:
def run_pipeline(path, save_mesh=True):
    '''Полный проход. Возвращает (mesh, per_segment_rows, summary).'''
    coords, labels = read_ply_cloud(path)
    coords, labels, _, _ = clean_and_normalize(coords, labels)
    segments = segments_by_label(coords, labels, min_points=MIN_SEGMENT_POINTS)
    if not segments:
        raise ValueError("не осталось сегментов после фильтрации")

    meshes, rows = [], []
    for seg_id, seg_pts in segments.items():
        seg_pts = subsample_segment(seg_pts, seed=int(seg_id))
        feats = extract_geometry_features(seg_pts)
        seg_type = infer_segment_type(feats)
        method, params = pick_reconstruction(seg_type, feats)
        mesh = build_mesh(seg_pts, method, params)
        metrics = score_mesh(seg_pts, mesh) if mesh else dict(EMPTY_METRICS)

        rows.append({
            "segment": seg_id, "n_points": len(seg_pts),
            "type": seg_type, "method": method,
            "linearity": feats["linearity"], "planarity": feats["planarity"],
            "sphericity": feats["sphericity"],
            "curvature_mean": feats["curvature_mean"],
            "normal_consistency": feats["normal_consistency"],
            "n_components": feats["n_components"],
            **metrics,
        })
        if mesh is not None:
            meshes.append(mesh)
        del mesh

    final = combine_meshes(meshes)
    del meshes
    if save_mesh and len(final.vertices) > 0:
        o3d.io.write_triangle_mesh(
            str(MESHES_DIR / (Path(path).stem + "_recon.ply")), final)

    global_metrics = score_mesh(coords, final)
    summary = {
        "file": Path(path).name,
        "n_segments": len(segments),
        "global_rmse": global_metrics["rmse"],
        "global_hausdorff": global_metrics["hausdorff_p95"],
        "global_chamfer": global_metrics["chamfer"],
        "global_artifacts": global_metrics["n_artifacts"],
    }
    return final, rows, summary

## 11. Прогон по всей папке

In [ ]:
import csv
import gc
import subprocess
import sys

files_to_process = cloud_files if PROCESS_ALL else cloud_files[:DEBUG_MAX_FILES]
print(f"К обработке: {len(files_to_process)} из {len(cloud_files)} файлов")

WORKER = Path("task2_process_one.py")
if not WORKER.exists():
    WORKER = Path("sem2/task2_process_one.py")
WORKER = WORKER.resolve()
assert WORKER.exists(), f"Не найден worker: {WORKER}"

CHECKPOINT_PATH = RESULT_DIR / "processed_files.txt"
PROGRESS_CSV = RESULT_DIR / "segments_report.csv"
SUMMARIES_PATH = RESULT_DIR / "summaries.json"

done_files = set()
if CHECKPOINT_PATH.exists():
    done_files = {ln.strip() for ln in CHECKPOINT_PATH.read_text().splitlines() if ln.strip()}
    print(f"Чекпоинт: уже готово {len(done_files)} файлов — продолжаем с места остановки")

segment_records = []
if PROGRESS_CSV.exists() and done_files:
    with open(PROGRESS_CSV, newline="") as fh:
        segment_records = list(csv.DictReader(fh))

file_summaries = []
if SUMMARIES_PATH.exists() and done_files:
    with open(SUMMARIES_PATH) as fh:
        file_summaries = json.load(fh)

start = time.time()
new_count = 0
worker_timeout = 600  # сек на один файл

for path in files_to_process:
    if path.name in done_files:
        continue

    n_done = len(done_files) + 1
    if new_count == 0 or n_done % 25 == 0:
        print(f"  [{n_done}/{len(files_to_process)}] {path.name}  "
              f"(прошло {time.time() - start:.1f}s)")

    cmd = [sys.executable, str(WORKER), str(path)]
    if len(done_files) < SAVE_MESH_LIMIT:
        cmd.append("--save-mesh")

    try:
        proc = subprocess.run(
            cmd, capture_output=True, text=True, timeout=worker_timeout,
            cwd=str(WORKER.parent),
        )
    except subprocess.TimeoutExpired:
        print(f"    !!! таймаут на {path.name}")
        continue

    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "")[-400:]
        print(f"    !!! сбой на {path.name} (код {proc.returncode}): {tail}")
        continue

    try:
        payload = json.loads(proc.stdout.strip().splitlines()[-1])
    except (json.JSONDecodeError, IndexError) as err:
        print(f"    !!! неверный ответ для {path.name}: {err}")
        continue

    rows = payload.get("rows", [])
    summary = payload.get("summary", {})
    for r in rows:
        r["file"] = path.name
    segment_records.extend(rows)
    file_summaries.append(summary)
    done_files.add(path.name)
    new_count += 1

    with open(CHECKPOINT_PATH, "a") as fh:
        fh.write(path.name + "\n")

    if new_count % 5 == 0:
        with open(PROGRESS_CSV, "w", newline="") as fh:
            if segment_records:
                writer = csv.DictWriter(fh, fieldnames=list(segment_records[0].keys()))
                writer.writeheader()
                writer.writerows(segment_records)
        with open(SUMMARIES_PATH, "w") as fh:
            json.dump(file_summaries, fh, indent=2)

print(f"\nГотово. Сегментов всего: {len(segment_records)}. "
      f"Время: {time.time() - start:.1f}s")

csv_path = RESULT_DIR / "segments_report.csv"
if segment_records:
    with open(csv_path, "w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(segment_records[0].keys()))
        writer.writeheader()
        writer.writerows(segment_records)
    print("CSV с деталями:", csv_path)

with open(RESULT_DIR / "summaries.json", "w") as fh:
    json.dump(file_summaries, fh, indent=2)
print("Сводки по файлам:", RESULT_DIR / "summaries.json")


## 12. Сводка по типам сегментов и методам

In [ ]:
type_counts = Counter(r["type"] for r in segment_records)
method_counts = Counter(r["method"] for r in segment_records)
print("Типы сегментов:", dict(type_counts))
print("Методы:        ", dict(method_counts))

rmse_by_type = defaultdict(list)
for r in segment_records:
    if np.isfinite(r["rmse"]):
        rmse_by_type[r["type"]].append(r["rmse"])

print("\nRMSE по типу сегмента:")
for t, values in rmse_by_type.items():
    print(f"  {t:>8s}: mean={np.mean(values):.4f}, "
          f"median={np.median(values):.4f}, N={len(values)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(type_counts.keys(), type_counts.values(), color="#2563eb")
axes[0].set_title("Сегменты по типам")
axes[0].set_ylabel("Кол-во")
axes[1].bar(method_counts.keys(), method_counts.values(), color="#16a34a")
axes[1].set_title("Выбранные методы")
axes[1].set_ylabel("Кол-во")
plt.tight_layout()
plt.savefig(RESULT_DIR / "type_method_distribution.png", dpi=120)
plt.show()

order = ["plane", "tube", "sphere", "complex"]
box_data = [rmse_by_type[t] for t in order if rmse_by_type[t]]
box_labels = [t for t in order if rmse_by_type[t]]
plt.figure(figsize=(8, 4))
plt.boxplot(box_data, labels=box_labels)
plt.ylabel("RMSE")
plt.title("RMSE по типам сегментов")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / "rmse_by_type.png", dpi=120)
plt.show()

## 13. Сравнение всех методов на одинаковых сегментах

Берём по одному сегменту каждого типа и применяем к ним **все три** алгоритма, чтобы
проверить, оправдано ли адаптивное переключение между Poisson / Alpha / Ball Pivoting.

In [ ]:
if SKIP_METHOD_COMPARISON:
    print("SKIP_METHOD_COMPARISON=True — сравнение всех методов пропущено (FAST_MODE)")
    comparison = []
else:
    def try_all_methods(seg, density):
        scored = {}
        for name, params in [
            ("poisson", {"depth": POISSON_DEPTH}),
            ("alpha_shape", {"alpha": max(density * 2.5, 0.01)}),
            ("ball_pivoting", {"radii": [density * 1.5, density * 2.5, density * 4.0]}),
        ]:
            scored[name] = score_mesh(seg, build_mesh(seg, name, params))
        return scored

    samples = {}
    for r in segment_records:
        if r["type"] not in samples and r["n_points"] > 200:
            samples[r["type"]] = r
        if len(samples) == 4:
            break

    comparison = []
    for seg_type, r in samples.items():
        coords, labels = read_ply_cloud(CLOUDS_DIR / r["file"])
        coords, labels, _, _ = clean_and_normalize(coords, labels)
        seg_pts = coords[labels == r["segment"]]
        if len(seg_pts) < MIN_SEGMENT_POINTS:
            continue
        feats = extract_geometry_features(seg_pts)
        for method, metrics in try_all_methods(seg_pts, feats["density"]).items():
            comparison.append({"type": seg_type, "method": method, **metrics})

    print(f'\n{"Type":>8s} | {"Method":>14s} | {"RMSE":>8s} | '
          f'{"Hausdorff":>10s} | {"Chamfer":>8s} | {"Artifacts":>9s}')
    print("-" * 80)
    for r in comparison:
        rmse = f'{r["rmse"]:.4f}' if np.isfinite(r["rmse"]) else "inf"
        hd = f'{r["hausdorff_p95"]:.4f}' if np.isfinite(r["hausdorff_p95"]) else "inf"
        ch = f'{r["chamfer"]:.4f}' if np.isfinite(r["chamfer"]) else "inf"
        print(f'{r["type"]:>8s} | {r["method"]:>14s} | {rmse:>8s} | '
              f'{hd:>10s} | {ch:>8s} | {r["n_artifacts"]:>9d}')

## 14. Визуализация: исходное облако и реконструкция

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

if SKIP_VISUALIZATION:
    print("SKIP_VISUALIZATION=True — 3D-визуализация пропущена (Open3D в kernel нестабилен)")
else:
    import subprocess
    example_path = files_to_process[0]
    coords, labels = read_ply_cloud(example_path)
    coords, labels, _, _ = clean_and_normalize(coords, labels)

    proc = subprocess.run(
        [sys.executable, str(WORKER), str(example_path), "--save-mesh"],
        capture_output=True, text=True, timeout=600,
        cwd=str(WORKER.parent),
    )
    summary = {}
    if proc.returncode == 0:
        payload = json.loads(proc.stdout.strip().splitlines()[-1])
        summary = payload.get("summary", {})

    fig = plt.figure(figsize=(14, 6))
    ax1 = fig.add_subplot(121, projection="3d")
    ax1.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=labels, cmap="tab20", s=1)
    ax1.set_title(f"Исходное облако (по сегментам)\n{example_path.name}")
    ax1.set_axis_off()

    ax2 = fig.add_subplot(122, projection="3d")
    mesh_path = MESHES_DIR / (example_path.stem + "_recon.ply")
    if mesh_path.exists():
        recon_mesh = o3d.io.read_triangle_mesh(str(mesh_path))
        verts = np.asarray(recon_mesh.vertices)
        tris = np.asarray(recon_mesh.triangles)
        if len(verts) > 0:
            ax2.plot_trisurf(verts[:, 0], verts[:, 1], tris, verts[:, 2],
                             cmap="viridis", alpha=0.85, linewidth=0)
            rmse = summary.get("global_rmse", float("nan"))
            ax2.set_title(f'Реконструкция\n{summary.get("n_segments", "?")} сегм., RMSE={rmse:.4f}')
        else:
            ax2.set_title("Реконструкция пуста")
    else:
        ax2.set_title("Mesh не сохранён")
    ax2.set_axis_off()
    plt.tight_layout()
    plt.savefig(RESULT_DIR / "recon_example.png", dpi=120)
    plt.show()


## 15. Выводы

1. Собран полный pipeline: чтение → предобработка → сегментация по `label` →
   геометрический анализ → классификация → выбор метода → реконструкция → сборка → метрики.
2. Тип сегмента определяется по собственным значениям ковариационной матрицы (PCA)
   и согласованности нормалей — это устойчиво разделяет плоскости / трубы / сферы / сложные формы.
3. Под каждый тип подобран свой алгоритм:
   - **plane** → Alpha Shape (аккуратно обрезает границы),
   - **tube** → Ball Pivoting (несколько радиусов под разную плотность),
   - **sphere/complex** → Poisson (замкнутая поверхность).
4. Кроме RMSE считаются Hausdorff-p95, Chamfer и число артефактов — это даёт более полную
   картину качества, чем одна метрика.
5. Сравнение всех методов на одинаковых сегментах (раздел 13) подтверждает, что адаптивный
   выбор алгоритма выигрывает у единого метода на весь датасет.